# Parking Space Detection with YOLOv8

**Goal:** Detect each parking space in a camera image or video and classify it as **empty** or **occupied**.

**Pipeline:** PKLot dataset → train YOLOv8n → evaluate → test on images → measure speed → video demo → simulated IoT message

**Before you start (Google Colab):** go to *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom.

## Step 1: Install the libraries

In [ ]:
!pip install -q ultralytics roboflow opencv-python-headless

import ultralytics
ultralytics.checks()   # shows the Python, torch and GPU versions

## Step 2: Download the PKLot dataset (YOLO format)

1. Create a free account at [roboflow.com](https://roboflow.com).
2. Open the PKLot page: https://public.roboflow.com/object-detection/pklot
3. Choose a version, click **Download → YOLOv8 → Show download code**.
4. Copy your **API key**, **workspace**, **project** and **version** into the cell below.

*Tip:* the full dataset is about 12,000 images. If training is too slow, pick a smaller version (for example the 640 resized one).

In [ ]:
from roboflow import Roboflow

API_KEY   = "PASTE_YOUR_API_KEY_HERE"
WORKSPACE = "brad-dwyer"      # replace with the value from the download code
PROJECT   = "pklot-1tros"     # replace with the value from the download code
VERSION   = 2                 # replace with the value from the download code

rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")

DATA_DIR  = dataset.location
DATA_YAML = f"{DATA_DIR}/data.yaml"
print("Dataset saved in:", DATA_DIR)

## Step 3: Explore the dataset

Check the class names (labels), count the images, and draw the labels on one image.

In [ ]:
import yaml, glob, os

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print("Classes:", cfg["names"])

for split in ["train", "valid", "test"]:
    n = len(glob.glob(f"{DATA_DIR}/{split}/images/*"))
    print(f"{split:5s}: {n} images")

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

img_path = random.choice(glob.glob(f"{DATA_DIR}/train/images/*"))
lbl_path = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"

img = cv2.imread(img_path)
h, w = img.shape[:2]
names = cfg["names"]

# YOLO label line = class x_center y_center width height (all 0-1)
with open(lbl_path) as f:
    for line in f:
        c, xc, yc, bw, bh = map(float, line.split())
        x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
        x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
        name = names[int(c)]
        color = (0, 255, 0) if "empty" in name else (0, 0, 255)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Green = empty, Red = occupied")
plt.axis("off"); plt.show()

## Step 4: Train YOLOv8n

- **yolov8n** = the *nano* model: small and fast, good for IoT devices.
- **Transfer learning:** we start from weights already trained on the COCO dataset.
- **epochs** = how many times the model sees all the training data.
- YOLO uses **gradient descent** (SGD/AdamW) and **backpropagation** to update its weights, as studied in class.
- The `hsv_v` (brightness) and `degrees` settings add **data augmentation**, which helps with shadows, rain and lighting changes.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=20,          # start with 20; increase later if accuracy is still improving
    imgsz=640,          # image size; smaller = faster
    batch=16,
    hsv_v=0.5,          # brightness augmentation (lighting / shadows)
    degrees=5,          # small rotations (camera angle)
    patience=5,         # early stopping if no improvement for 5 epochs
    project="runs",
    name="pklot_yolov8n",
)

## Step 5: Evaluate the model

- **Precision:** of the spaces the model called occupied, how many really were.
- **Recall:** of the spaces that really were occupied, how many the model found.
- **mAP50:** the main object-detection score (0 to 1, higher is better).
- The **confusion matrix** shows where the model mixes up empty and occupied.

In [ ]:
best = YOLO("runs/pklot_yolov8n/weights/best.pt")
metrics = best.val(data=DATA_YAML, split="test")

print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")
print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")

In [ ]:
from IPython.display import Image, display

run = "runs/pklot_yolov8n"
for f in ["results.png", "confusion_matrix.png"]:
    p = f"{run}/{f}"
    if os.path.exists(p):
        print(f); display(Image(p, width=800))

## Step 6: Test on images and count free spaces

In [ ]:
def count_spaces(result):
    """Count empty and occupied spaces in one YOLO result."""
    free = occupied = 0
    for c in result.boxes.cls.tolist():
        name = result.names[int(c)]
        if "empty" in name:
            free += 1
        elif "occupied" in name:
            occupied += 1
    return free, occupied

test_imgs = random.sample(glob.glob(f"{DATA_DIR}/test/images/*"), 3)
for p in test_imgs:
    r = best.predict(p, conf=0.4, verbose=False)[0]
    free, occ = count_spaces(r)
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    plt.title(f"Free: {free}   Occupied: {occ}")
    plt.axis("off"); plt.show()

## Step 7: Measure speed (less time-consuming detection)

**FPS** = frames per second. A higher number means faster detection. Report this in your presentation.

In [ ]:
import time

imgs = glob.glob(f"{DATA_DIR}/test/images/*")[:50]
best.predict(imgs[0], verbose=False)       # warm-up run

start = time.time()
for p in imgs:
    best.predict(p, verbose=False)
elapsed = time.time() - start

print(f"Average time per image: {elapsed / len(imgs) * 1000:.1f} ms")
print(f"Speed: {len(imgs) / elapsed:.1f} FPS")

## Step 8: Video demo

Upload a parking lot video (for example `parking.mp4`) using the Colab file panel on the left.

To save time, we only run detection **every N frames**, because cars in a parking lot don't move quickly. The other frames reuse the last result.

In [ ]:
VIDEO_IN  = "parking.mp4"
VIDEO_OUT = "parking_result.mp4"
DETECT_EVERY = 15     # run YOLO every 15 frames (about twice per second at 30 FPS)

cap = cv2.VideoCapture(VIDEO_IN)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

frame_no, last = 0, None
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if frame_no % DETECT_EVERY == 0:
        last = best.predict(frame, conf=0.4, verbose=False)[0]
    if last is not None:
        free, occ = count_spaces(last)
        for box, c in zip(last.boxes.xyxy.tolist(), last.boxes.cls.tolist()):
            x1, y1, x2, y2 = map(int, box)
            color = (0, 255, 0) if "empty" in last.names[int(c)] else (0, 0, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.rectangle(frame, (10, 10), (330, 60), (0, 0, 0), -1)
        cv2.putText(frame, f"Free: {free}  Occupied: {occ}", (20, 45),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    out.write(frame)
    frame_no += 1

cap.release(); out.release()
print("Saved:", VIDEO_OUT, "- download it from the file panel")

## Step 9: Simulated IoT message

In a real system, a small device (Raspberry Pi / Jetson Nano) next to the camera runs the model and sends a short message to a dashboard or app, usually with the **MQTT** protocol.

Here we **simulate** that: we build the JSON message the device would send. Only the counts are sent, not the video, which saves network bandwidth and protects privacy.

In [ ]:
import json
from datetime import datetime

r = best.predict(random.choice(imgs), conf=0.4, verbose=False)[0]
free, occ = count_spaces(r)

message = {
    "lot_id": "LOT-A",
    "camera_id": "CAM-01",
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "free_spaces": free,
    "occupied_spaces": occ,
    "occupancy_rate": round(occ / max(free + occ, 1), 2),
}
print("MQTT topic: parking/LOT-A/status")
print(json.dumps(message, indent=2))

# Real MQTT version (optional, needs a broker such as test.mosquitto.org):
# !pip install -q paho-mqtt
# import paho.mqtt.publish as publish
# publish.single("parking/LOT-A/status", json.dumps(message), hostname="test.mosquitto.org")

## Step 10: Save the model for IoT devices

Export to **ONNX**, a format that runs on many small devices (Raspberry Pi, Jetson).

In [ ]:
best.export(format="onnx", imgsz=640)
print("Exported: runs/pklot_yolov8n/weights/best.onnx")

## Next steps

1. **Test on a new parking lot:** run the model on CNRPark-EXT images to see how well it works on camera angles it has never seen.
2. **Segmentation comparison:** train `yolov8n-seg.pt` using polygon labels (download PKLot in *YOLOv8 Instance Segmentation* format if available).
3. **Classification comparison:** crop each space and train a small CNN (MobileNetV2) to label it empty or occupied.
4. **Results table for the presentation:** precision, recall, mAP50 and FPS for each approach.